# Segment teeth on intraoral scans

This notebook will guide you on how to train a segmentation model on dental scans and label every vertex of a scan with its tooth number, from `.obj` meshes in to per-vertex labels out.

The data is [Teeth3DS](https://github.com/abenhamadou/3DTeethSeg22_challenge), the benchmark of the 3DTeethSeg 2022 challenge: 1800 intraoral scans of upper and lower jaws from 900 patients, each vertex annotated with its tooth in the FDI numbering and with a tooth instance. The job is the one an orthodontic planning tool does first, and every published method on it comes with its own mesh loader, sampler and training script. Here it is a dataset class, a transform pipeline and the same training loop as the other tutorials.

## Setup

In [ ]:
# On Colab:
# !pip install "torch-pointcloud"
# !pip install pyg-lib -f https://data.pyg.org/whl/torch-2.10.0+cu128.html

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import torch

import torch_pointcloud as tp
import torch_pointcloud.transforms as T
from torch_pointcloud.ops.geometry import vertex_normals

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch-pointcloud", tp.__version__, "| device:", DEVICE)

## Helpers

A few plotting helpers used throughout the notebook. A jaw is looked at from the crowns, so the viewpoint flips with the jaw's orientation in the scanner frame.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Transparent canvas and gray ink, so the same figure reads on a light and on a dark page.
INK = "#7a7a7a"
plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "none", "axes.facecolor": "none", "savefig.transparent": True,
                     "text.color": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "axes.edgecolor": INK})


def figure(columns, height=5.0):
    """A figure of `columns` 3D panels side by side, with the margins a 3D axes wastes taken back."""
    fig = plt.figure(figsize=(12, height))
    fig.subplots_adjust(left=0.0, right=1.0, top=0.95, bottom=0.08, wspace=0.0, hspace=0.0)
    return fig


def show_scan(pos, color, *, ax=None, title=None, size=0.8, elev=70, azim=-90, cmap=None, zoom=1.3):
    """Scatter a scan: `color` is one color per vertex, or one scalar per vertex drawn with `cmap`."""
    if ax is None:
        ax = figure(1).add_subplot(projection="3d")
    p = pos.cpu().numpy()
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=color, s=size, linewidths=0, depthshade=False, cmap=cmap)
    ax.view_init(elev=elev, azim=azim)
    ax.set_box_aspect(np.ptp(p, axis=0), zoom=zoom)
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax


def crown_side(normal, labels):
    """The elevation that looks at the crowns: from above when the teeth point up, from below otherwise."""
    return 70 if float(normal[labels > 0, 2].mean()) >= 0 else -70


def class_legend(fig, names, colors):
    """One shared key under the panels of a figure."""
    fig.legend(handles=[Patch(color=c, label=n) for n, c in zip(names, colors)], loc="lower center", ncol=len(names), frameon=False, fontsize=9)


def bar_chart(ax, names, values, colors, ylabel, limit=None):
    """Labeled bars, each with its value written above it."""
    ax.bar(names, values, color=colors, width=0.62)
    for i, v in enumerate(values):
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.set(ylabel=ylabel, ylim=(0, limit))
    ax.spines[["top", "right"]].set_visible(False)

## Data

Teeth3DS is distributed as six archives on [OSF](https://osf.io/xctdy/), under a CC BY-NC-ND 4.0 licence. Extract them under `data/Teeth3DS/raw/`, so that a scan sits at `raw/<jaw>/<patient>/<patient>_<jaw>.obj` next to its `.json`, and put the four split files of the official Teeth3DS split under `raw/splits/`. The OBJ carries the vertex positions in millimeters and the faces; only a third of the files also carry vertex normals, so the reader computes them from the faces for every scan with `vertex_normals`. The JSON carries one FDI label and one instance id per vertex, with 0 for the gingiva.

In [ ]:
ROOT = Path("data")


def read_scan(obj_path):
    """Read one intraoral scan: vertex positions from the OBJ, normals from its faces, FDI labels and instances from the JSON."""
    obj_path = Path(obj_path)
    vertices, faces = [], []
    with open(obj_path) as f:
        for line in f:
            if line.startswith("v "):
                vertices.append(line.split()[1:4])
            elif line.startswith("f "):
                faces.append([token.split("/")[0] for token in line.split()[1:4]])
    pos = torch.from_numpy(np.asarray(vertices, dtype=np.float32))
    face = torch.from_numpy(np.asarray(faces, dtype=np.int64) - 1)
    meta = json.loads(obj_path.with_suffix(".json").read_text())
    return {
        "pos": pos,
        "normal": vertex_normals(pos, face),
        "segment": torch.tensor(meta["labels"], dtype=torch.int64),
        "instance": torch.tensor(meta["instances"], dtype=torch.int64),
    }


scan = read_scan(ROOT / "Teeth3DS" / "raw" / "lower" / "013FHA7K" / "013FHA7K_lower.obj")
pos = scan["pos"]
print(f"{len(pos):,} vertices, extent", [f"{float(v):.1f} mm" for v in pos.max(0).values - pos.min(0).values])
codes, counts = torch.unique(scan["segment"], return_counts=True)
print("FDI codes:", {int(c): int(n) for c, n in zip(codes, counts)})
print("teeth:", int(scan["instance"].max()))

A lower jaw of 107 000 vertices over 80 mm, with 13 teeth: both third molars are missing, which is the rule rather than the exception in this data, and so is the lower-right second premolar, code 45. The FDI code of a tooth is its quadrant times ten plus its position from the midline, so `36` is the lower-left first molar.

The model predicts 33 classes: the gingiva and the 32 possible teeth. `Relabel` with a list maps each FDI code to its index in that list, and the list order is what the head's channels mean.

In [ ]:
FDI = [0] + [10 * quadrant + tooth for quadrant in (1, 2, 3, 4) for tooth in range(1, 9)]
CLASSES = ["gingiva"] + [f"tooth {code}" for code in FDI[1:]]
TOOTH_TYPES = ["incisor 1", "incisor 2", "canine", "premolar 1", "premolar 2", "molar 1", "molar 2", "molar 3"]
TYPE_COLOR = np.array(["#c4c4c4", "#ff5722", "#2f7bbf", "#3fa66a", "#a862c4", "#e0a600", "#00a3a3", "#d6455a", "#8a6a3c"])  # gingiva, then the eight tooth types


def tooth_type(labels):
    """Tooth type 1 to 8 of each class index, 0 for the gingiva."""
    return torch.tensor(FDI)[labels] % 10


labels = T.Relabel(keys="segment", labels=FDI)({"segment": scan["segment"]})["segment"]
print(len(CLASSES), "classes;", CLASSES[:3], "...")

In [ ]:
elev = crown_side(scan["normal"], labels)
fig = figure(2)
show_scan(pos, TYPE_COLOR[tooth_type(labels)], ax=fig.add_subplot(121, projection="3d"), title=f"013FHA7K_lower, {len(pos) / 1e3:.0f} k vertices by FDI annotation", elev=elev)
show_scan(pos, scan["normal"][:, 2] * np.sign(elev), ax=fig.add_subplot(122, projection="3d"), title="the same scan shaded by its vertex normals", elev=elev, cmap="viridis")
class_legend(fig, TOOTH_TYPES + ["gingiva"], TYPE_COLOR[[1, 2, 3, 4, 5, 6, 7, 8, 0]])

![A scan by its FDI annotation and by its vertex normals](../assets/tutorials/teeth_scan.png)

## From meshes to a cached dataset

Parsing an OBJ is the slow part, so the dataset does it once per scan and caches a `.pt` under `processed/`, in parallel. `PointCloudDataset` provides the `raw/` and `processed/` paths; the split files name the scans.

In [ ]:
from multiprocessing import Pool

from tqdm.auto import tqdm

from torch_pointcloud.datasets import PointCloudDataset


def _process_one(args):
    obj_path, out_path = args
    torch.save(read_scan(obj_path), out_path)


class Teeth3DS(PointCloudDataset):
    def __init__(self, root, split="training", transform=None, num_workers=8):
        super().__init__(root)
        self.split = split
        self.transform = transform
        names = []
        for jaw in ("upper", "lower"):
            names += (Path(self.raw_dir) / "splits" / f"{split}_{jaw}.txt").read_text().split()
        self.names = sorted(names)
        Path(self.processed_dir).mkdir(parents=True, exist_ok=True)
        todo = [name for name in self.names if not (Path(self.processed_dir) / f"{name}.pt").exists()]
        if todo:
            jobs = [(Path(self.raw_dir) / name.split("_")[1] / name.split("_")[0] / f"{name}.obj", Path(self.processed_dir) / f"{name}.pt") for name in todo]
            with Pool(num_workers) as pool:
                list(tqdm(pool.imap_unordered(_process_one, jobs), total=len(jobs), desc=f"Caching {split} scans"))

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        data = torch.load(Path(self.processed_dir) / f"{self.names[i]}.pt", weights_only=True)
        data["name"] = self.names[i]
        return self.transform(data) if self.transform is not None else data


start = time.perf_counter()
train_dataset = Teeth3DS(ROOT, "training")
test_dataset = Teeth3DS(ROOT, "testing")
print(f"train scans: {len(train_dataset):,} | test scans: {len(test_dataset):,} | cached in {time.perf_counter() - start:.0f} s")

1200 training scans and 600 test scans. The first pass parses every OBJ, a few minutes with eight workers; every later one reads the cache. Scans hold between 26 k and 185 k vertices, 120 k in the median.

## The scan pipeline

- **Labels** map FDI codes to the 33 classes with `Relabel`.
- **Coordinates** are centered on the scan's centroid with `Shift`. They stay in millimeters: the model's radii below are metric too.
- **Features** are the vertex normals, which is what separates a crown from the gum at a boundary. `Cat` puts them under `x`.
- **A 0.5 mm grid** (`Voxelize`) brings every scan to the same density, about a quarter of the vertices, and records the inverse map so that predictions come back to every vertex of the mesh.
- **Augmentation**, in the train pipeline only, is a rotation of up to 15 degrees about each axis, a scale of 10 % and a small jitter. No flips: a mirrored jaw swaps the left and right quadrants, and with them the labels.

`RandomRotate` takes both `pos` and `x` here, so the normals turn with the mesh.

In [ ]:
GRID = 0.5  # mm

features = T.Compose([
    T.Relabel(keys="segment", labels=FDI, allow_missing_keys=True),
    T.Shift(keys="pos", method="centroid"),
    T.Cat(keys=["normal"], dst_key="x", dim=1),
])
eval_transform = T.Compose([
    features,
    T.Voxelize(pos_key="pos", pos_reduce="mean", size=GRID, keys=["x", "segment"], dst_inverse_key="inverse", allow_missing_keys=True),
    T.KeepItems(keys=["pos", "x", "segment", "inverse", "name"], allow_missing_keys=True),
])
train_transform = T.Compose([
    features,
    T.Voxelize(pos_key="pos", pos_reduce="mean", size=GRID, keys=["x", "segment"], random_first=True),
    T.RandomRotate(keys=["pos", "x"], angle_range=(-15.0, 15.0), axis=0),
    T.RandomRotate(keys=["pos", "x"], angle_range=(-15.0, 15.0), axis=1),
    T.RandomRotate(keys=["pos", "x"], angle_range=(-15.0, 15.0), axis=2),
    T.RandomScale(keys="pos", scale_range=(0.9, 1.1)),
    T.RandomJitter(keys="pos", sigma=0.05, clip=0.2),
    T.KeepItems(keys=["pos", "x", "segment"]),
])

sample = eval_transform(train_dataset[0])
print({k: tuple(v.shape) for k, v in sample.items() if torch.is_tensor(v)})

## Model

PointNeXt-S, the configuration the library ships for S3DIS, with two changes: three input channels for the normals, and radii in millimeters. The grouping radius of the first stage is 1.5 mm, which at the 0.5 mm grid holds about 30 neighbors, and each stage doubles it up to 24 mm, a molar's width.

In [ ]:
from torch_pointcloud.models import PointNeXtSegmentation

model = PointNeXtSegmentation(
    in_channels=3,
    num_classes=len(CLASSES),
    stem_channels=32,
    stem_plain_last=True,
    encoder_channels=[64, 128, 256, 512],
    encoder_depths=[0, 0, 0, 0],
    encoder_expansion=4,
    sa_layers=2,
    sa_use_res=True,
    decoder_channels=[256, 128, 64, 32],
    decoder_depths=[2, 2, 2, 2],
    decoder_plain_last=False,
    head_channels=[32],
    ratios=[0.25, 0.25, 0.25, 0.25],
    radiuses=[1.5, 3.0, 6.0, 12.0, 24.0],
    num_neighbors=[32, 32, 32, 32, 32],
).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

In [ ]:
from torch_pointcloud.utils.data import PointCloudDataLoader

BATCH_SIZE, EPOCHS, LR = 8, 20, 2e-3

train_dataset.transform, test_dataset.transform = train_transform, eval_transform
train_loader = PointCloudDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=6)
test_loader = PointCloudDataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=6)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, total_steps=EPOCHS * len(train_loader), pct_start=0.1)
print(len(train_loader), "steps per epoch")

## Training

The loop is the one of the other tutorials. A batch is eight scans on the 0.5 mm grid, about 150 000 points; validation runs on the test split at grid resolution after each epoch.

In [ ]:
from torch_pointcloud.metrics import accuracy, confusion_matrix, intersection_over_union


def train_one_epoch(model, dataloader, optimizer, scheduler, criterion):
    model.train()
    total_loss, total_points = 0.0, 0
    for data in tqdm(dataloader, desc="Training", leave=False):
        x, pos, batch, target = (data[k].to(DEVICE) for k in ("x", "pos", "batch", "segment"))
        optimizer.zero_grad()
        loss = criterion(model(x, pos, batch), target)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * len(target)
        total_points += len(target)
    return total_loss / total_points


@torch.no_grad()
def evaluate(model, dataloader):
    model.eval()
    cm = torch.zeros(len(CLASSES), len(CLASSES), dtype=torch.long)
    for data in tqdm(dataloader, desc="Evaluating", leave=False):
        x, pos, batch, target = (data[k].to(DEVICE) for k in ("x", "pos", "batch", "segment"))
        preds = model(x, pos, batch).argmax(dim=1)
        cm += confusion_matrix(preds.cpu(), target.cpu(), len(CLASSES))
    return cm


history, best = [], None
for epoch in range(EPOCHS):
    start = time.perf_counter()
    loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion)
    cm = evaluate(model, test_loader)
    miou = intersection_over_union(cm)
    history.append({"epoch": epoch + 1, "loss": loss, "miou": miou, "iou": intersection_over_union(cm, average="none", class_names=CLASSES)})
    print(f"epoch {epoch + 1:2d}/{EPOCHS} | loss {loss:.3f} | test mIoU {miou:.3f} OA {accuracy(cm):.3f} | {time.perf_counter() - start:.0f} s")
    if best is None or miou > best["miou"]:
        best = history[-1]
        torch.save(model.state_dict(), "pointnext_teeth3ds.pt")


print(f"best test mIoU {best['miou']:.3f} at epoch {best['epoch']} | peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
print("best epoch per-class IoU:", {name: round(value, 3) for name, value in best["iou"].items()})

An epoch is 150 steps, about 25 s with the evaluation on an RTX 3090 Ti, at 2.3 GiB. The best mean IoU over the 33 classes is 0.769 at epoch 19. Averaged per tooth type:

| tooth type | IoU |
| --- | --- |
| central incisor | 0.90 |
| lateral incisor | 0.91 |
| canine | 0.91 |
| first premolar | 0.90 |
| second premolar | 0.84 |
| first molar | 0.85 |
| second molar | 0.78 |
| third molar | 0.02 |

Third molars are the hard class: half of the jaws do not have them, and when they do the model reads them as second molars for most of their vertices. Everything else confuses with its neighbor along the arch, the second molar with the first, which the per-vertex accuracy hides and the per-class IoU shows.

In [ ]:
by_type = {t: [] for t in range(1, 9)}
for name, value in best["iou"].items():
    index = CLASSES.index(name)
    if index > 0:
        by_type[FDI[index] % 10].append(value)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([e["epoch"] for e in history], [e["loss"] for e in history], marker="o", label="training loss")
axes[0].plot([e["epoch"] for e in history], [e["miou"] for e in history], marker="o", label="test mIoU")
axes[0].set(xlabel="epoch", xticks=[e["epoch"] for e in history][::2], title=f"{EPOCHS} epochs over the training scans")
axes[0].legend(frameon=False)
axes[0].spines[["top", "right"]].set_visible(False)
bar_chart(axes[1], TOOTH_TYPES, [np.mean(by_type[t]) for t in range(1, 9)], TYPE_COLOR[1:], "IoU", limit=1)
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_title(f"test IoU per tooth type at the best epoch, mean {best['miou']:.2f}")

![Training curves and IoU per tooth type](../assets/tutorials/teeth_training.png)

## Label every vertex

The model predicts on the grid; `inverse` carries each prediction back to every vertex of the mesh, so the test score below is at the resolution the annotation was made at.

In [ ]:
model.load_state_dict(torch.load("pointnext_teeth3ds.pt", map_location=DEVICE, weights_only=True))
model.eval()


@torch.no_grad()
def segment_scan(model, scan):
    data = eval_transform(scan)
    batch = torch.zeros(len(data["pos"]), dtype=torch.long, device=DEVICE)
    preds = model(data["x"].to(DEVICE), data["pos"].to(DEVICE), batch).argmax(dim=1).cpu()
    return preds[data["inverse"]]


predicted = segment_scan(model, scan)
wrong = predicted != labels
print("vertices correct:", f"{100 * float((~wrong).float().mean()):.2f}%")
print("teeth predicted:", int((predicted.unique() > 0).sum()), "| annotated:", int((labels.unique() > 0).sum()))

In [ ]:
fig = figure(3, height=4.2)
show_scan(pos, TYPE_COLOR[tooth_type(predicted)], ax=fig.add_subplot(131, projection="3d"), title="predicted, every vertex", elev=elev)
show_scan(pos, TYPE_COLOR[tooth_type(labels)], ax=fig.add_subplot(132, projection="3d"), title="the annotation", elev=elev)
show_scan(pos, np.where(wrong, "#ff5722", "#c4c4c4"), ax=fig.add_subplot(133, projection="3d"), title=f"the {100 * float(wrong.float().mean()):.1f}% of vertices that differ, in orange", elev=elev)
class_legend(fig, TOOTH_TYPES + ["gingiva"], TYPE_COLOR[[1, 2, 3, 4, 5, 6, 7, 8, 0]])

![Predicted labels, the annotation, and the vertices that differ](../assets/tutorials/teeth_prediction.png)

The differences sit on the gum line around each crown, where the annotators themselves drew the boundary by hand. The loop below scores the whole test split the same way, one scan per batch: a scan's `inverse` indexes its own grid points, and packing several scans would need the maps offset into the batch.

In [ ]:
cm = torch.zeros(len(CLASSES), len(CLASSES), dtype=torch.long)
per_scan = {}
start = time.perf_counter()
for data in tqdm(PointCloudDataLoader(test_dataset, batch_size=1, num_workers=6), desc="Test split at full resolution"):
    with torch.no_grad():
        preds = model(data["x"].to(DEVICE), data["pos"].to(DEVICE), data["batch"].to(DEVICE)).argmax(dim=1).cpu()
    full, target = preds[data["inverse"]], data["segment"][data["inverse"]]
    cm += confusion_matrix(full, target, len(CLASSES))
    per_scan[data["name"][0]] = float((full == target).float().mean())

print(f"mIoU {intersection_over_union(cm):.3f} | OA {accuracy(cm):.3f} over {len(per_scan)} scans in {time.perf_counter() - start:.0f} s")
print(f"three quarters of the scans are above {100 * np.percentile(list(per_scan.values()), 25):.0f}% of vertices correct")
hardest = sorted(per_scan.items(), key=lambda kv: kv[1])[:2]
print("hardest scans:", [(name, round(value, 3)) for name, value in hardest])

Over the 600 test scans at full resolution: mIoU 0.747, overall accuracy 0.936, in 27 s. Three quarters of the scans are above 93 % of vertices correct; the tail is where the data is unusual.

In [ ]:
raw_scans = Teeth3DS(ROOT, "testing")  # no transform: the scans as read from the files

fig = figure(2, height=9)
for row, (name, accuracy_) in enumerate(hardest):
    hard_scan = raw_scans[raw_scans.names.index(name)]
    hard_labels = T.Relabel(keys="segment", labels=FDI)({"segment": hard_scan["segment"]})["segment"]
    hard_elev = crown_side(hard_scan["normal"], hard_labels)
    show_scan(hard_scan["pos"], TYPE_COLOR[tooth_type(segment_scan(model, hard_scan))], ax=fig.add_subplot(2, 2, 2 * row + 1, projection="3d"), title=f"{name}, predicted, {100 * accuracy_:.0f}% of vertices correct", elev=hard_elev)
    show_scan(hard_scan["pos"], TYPE_COLOR[tooth_type(hard_labels)], ax=fig.add_subplot(2, 2, 2 * row + 2, projection="3d"), title=f"{name}, the annotation", elev=hard_elev)
class_legend(fig, TOOTH_TYPES + ["gingiva"], TYPE_COLOR[[1, 2, 3, 4, 5, 6, 7, 8, 0]])

![The two hardest test scans, predicted and annotated](../assets/tutorials/teeth_errors.png)

The hardest scans are `01MAU84T_lower` and `MNIAB8K3_lower`, at 46 % and 54 %. On such scans the model places a tooth one position off, and every vertex of that tooth counts as wrong: the gum line is right, the number is not. A crowded arch or a missing tooth shifts the count, which is why the challenge scores tooth identification separately from segmentation.

## From vertices to teeth

A per-vertex label is one step from what a planning tool wants, one mesh per tooth. `torch.unique` on the labels gives the vertices of each tooth; a centroid and an oriented bounding box per label are a few lines of `torch`. Everything else of a deployment is on this page: `read_scan`, `eval_transform`, `segment_scan` and a state dict, with the library and `numpy` as the only dependencies.

In [ ]:
centroids = {}
for label in predicted.unique().tolist():
    if label > 0:
        centroids[CLASSES[label]] = pos[predicted == label].mean(dim=0)
for name, centroid in sorted(centroids.items()):
    print(f"  {name}: ({float(centroid[0]):6.1f}, {float(centroid[1]):6.1f}, {float(centroid[2]):6.1f}) mm")

In [ ]:
fig = plt.figure(figsize=(6.4, 3.2))
ax = fig.add_subplot(projection="3d")
ax.set_position((0, 0, 1, 1))
show_scan(pos, TYPE_COLOR[tooth_type(predicted)], ax=ax, size=1.4, elev=elev * 0.85, azim=-65, zoom=1.5)